In [2]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping


In [3]:
# 1. Load your dataset
df = pd.read_csv("heart.csv")
df


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1020,59,1,1,140,221,0,1,164,1,0.0,2,0,2,1
1021,60,1,0,125,258,0,0,141,1,2.8,1,1,3,0
1022,47,1,0,110,275,0,0,118,1,1.0,1,1,2,0
1023,50,0,0,110,254,0,0,159,0,0.0,2,0,2,1


In [4]:
X = df.drop('target', axis=1)
y = df['target']


In [5]:
# 2. Scale the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [8]:
# Save scaler
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)


In [7]:
# 3. Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)


In [9]:
from sklearn.utils import class_weight
weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(weights))


In [10]:
model = Sequential([
    Dense(32, activation='relu', input_dim=X_train.shape[1]),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

c:\Users\parul\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [12]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),

    Dense(32, activation='relu'),
    Dropout(0.2),

    Dense(1, activation='sigmoid')
])


In [14]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [16]:
# 5. Train the model with early stopping
es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
model.fit(X_train, y_train, epochs=50, batch_size=8, validation_split=0.2, callbacks=[es])

Epoch 1/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.8437 - loss: 0.3607 - val_accuracy: 0.8293 - val_loss: 0.4001
Epoch 2/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.8517 - loss: 0.3480 - val_accuracy: 0.8476 - val_loss: 0.3875
Epoch 3/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.8637 - loss: 0.3264 - val_accuracy: 0.8537 - val_loss: 0.3903
Epoch 4/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.8475 - loss: 0.3179 - val_accuracy: 0.8537 - val_loss: 0.3730
Epoch 5/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.9053 - loss: 0.2812 - val_accuracy: 0.8659 - val_loss: 0.3891
Epoch 6/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.8802 - loss: 0.3187 - val_accuracy: 0.8537 - val_loss: 0.3648
Epoch 7/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.8830 - loss: 0.3031 - val_accuracy: 0.8720 - val_loss: 0.3502
Epoch 8/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9168 - loss: 0.2440 - val_accuracy: 0.8659 - v

In [17]:
# 6. Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Model Accuracy on Test Set: {accuracy * 100:.2f}%")

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.8796 - loss: 0.2838
Model Accuracy on Test Set: 86.83%


In [18]:
from sklearn.metrics import confusion_matrix, classification_report

y_probs = model.predict(X_test)
y_pred = (y_probs >= 0.5).astype(int)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 120ms/step
[[83 19]
 [ 8 95]]
              precision    recall  f1-score   support

           0       0.91      0.81      0.86       102
           1       0.83      0.92      0.88       103

    accuracy                           0.87       205
   macro avg       0.87      0.87      0.87       205
weighted avg       0.87      0.87      0.87       205



In [19]:

# 7. Save the trained ANN model
model.save('heart_model.h5')
print("Deep Learning model saved as heart_model.h5 and scaler.pkl.")

Deep Learning model saved as heart_model.h5 and scaler.pkl.
